# CSE 182 Project 2: GBM39 Amplicon 1 Reconstruction

This notebook parses the GBM39 Amplicon 1 graph and cycles files, classifies the edges, builds plotting-ready data tables, and creates a clean final visualization of the reconstructed ecDNA amplicon structure.

## Project Overview

Project 2 focuses on interpreting amplicon reconstruction output. The graph file provides sequence edges and breakpoint edges with genomic coordinates and copy-number estimates. The cycles file provides interval and segment definitions, plus the cycles that describe candidate ecDNA paths through those segments.

For GBM39 Amplicon 1, the final visualization focuses on chr7 from 54,729,879 to 56,219,661.

## Imports

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import GBM39_Project2_final as workflow

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## File Paths

The raw graph and cycles files are parsed directly. The original PNG and PDF are kept as visual references only.

In [2]:
graph_path = Path("GBM39_amplicon1_graph.txt")
cycles_path = Path("GBM39_amplicon1_cycles.txt")
reference_png = Path("GBM39_amplicon1.png")
reference_pdf = Path("GBM39_amplicon1.pdf")

input_files = [graph_path, cycles_path, reference_png, reference_pdf]
pd.DataFrame({"file": [path.name for path in input_files], "exists": [path.exists() for path in input_files]})

,file,exists
0,GBM39_amplicon1_graph.txt,True
1,GBM39_amplicon1_cycles.txt,True
2,GBM39_amplicon1.png,True
3,GBM39_amplicon1.pdf,True


## Parsing Functions

The parsing functions are imported from `GBM39_Project2_final.py`. They were built from the earlier parser notebook and keep the workflow modular: graph parsing, cycles parsing, edge classification, plotting-table creation, and final figure generation are separate steps.

## Parsing Execution

The graph file has a `SequenceEdge` section and a `BreakpointEdge` section. The cycles file has interval rows, segment rows, and semicolon-delimited cycle rows.

In [3]:
graph_edges_df = workflow.parse_graph_file(graph_path)
intervals_df, cycle_segments_df, cycles_df = workflow.parse_cycles_file(cycles_path)

graph_edges_df.to_csv("graph_edges.csv", index=False)
cycle_segments_df.to_csv("cycle_segments.csv", index=False)
cycles_df.to_csv("cycles.csv", index=False)

pd.DataFrame({
    "table": ["graph_edges", "intervals", "cycle_segments", "cycles"],
    "rows": [len(graph_edges_df), len(intervals_df), len(cycle_segments_df), len(cycles_df)]
})

,table,rows
0,graph_edges,24
1,intervals,1
2,cycle_segments,4
3,cycles,2


## Parsed Data Preview

In [4]:
display(graph_edges_df[["graph_section", "edge_type", "raw_edge_string", "chr1", "pos1", "strand1", "chr2", "pos2", "strand2", "copy_number"]].head(12))
display(cycle_segments_df[["segment_id", "chr", "start", "end"]])
display(cycles_df[["cycle_id", "copy_count", "segment_list", "segment_count"]])

,graph_section,edge_type,raw_edge_string,chr1,pos1,strand1,chr2,pos2,strand2,copy_number
0,SequenceEdge,sequence,chr7:54729879-->chr7:54743010+,chr7,54729879,-,chr7,54743010,+,3.031595
1,SequenceEdge,sequence,chr7:54743011-->chr7:54830971+,chr7,54743011,-,chr7,54830971,+,3.313981
2,SequenceEdge,sequence,chr7:54830972-->chr7:55194959+,chr7,54830972,-,chr7,55194959,+,116.204610
3,SequenceEdge,sequence,chr7:55194960-->chr7:55220100+,chr7,55194960,-,chr7,55220100,+,3.167294
4,SequenceEdge,sequence,chr7:55220101-->chr7:55222712+,chr7,55220101,-,chr7,55222712,+,3.180103
5,SequenceEdge,sequence,chr7:55222713-->chr7:55676816+,chr7,55222713,-,chr7,55676816,+,116.217419
6,SequenceEdge,sequence,chr7:55676817-->chr7:55677805+,chr7,55676817,-,chr7,55677805,+,16.643945
7,SequenceEdge,sequence,chr7:55677806-->chr7:56117062+,chr7,55677806,-,chr7,56117062,+,116.217419
8,SequenceEdge,sequence,chr7:56117063-->chr7:56219661+,chr7,56117063,-,chr7,56219661,+,3.326789
9,BreakpointEdge,source,chr7:-1-->chr7:54729879-,chr7,-1,-,chr7,54729879,-,3.031595


,segment_id,chr,start,end
0,1,chr7,54830972,55194959
1,2,chr7,55222713,55676816
2,3,chr7,55222713,56117062
3,4,chr7,55677806,56117062


,cycle_id,copy_count,segment_list,segment_count
0,1,99.573474,"2+,4+,1+",3
1,2,13.317155,"3+,1+",2


## Edge Classification

Sequence edges are drawn as horizontal copy-number segments. Valid intrachromosomal discordant breakpoint edges are drawn as arcs. Source edges with placeholder coordinate `-1` are kept in the table but excluded from arc drawing.

In [5]:
classified_edges_df = workflow.classify_graph_edges(graph_edges_df)
classification_summary = classified_edges_df["plotting_category"].value_counts().rename_axis("plotting_category").reset_index(name="count")
classification_summary

,plotting_category,count
0,breakpoint_intrachromosomal,11
1,sequence_edge,9
2,unclear,4


## Plotting-Ready Data Creation

In [6]:
segment_plot_df = workflow.build_segment_plot_df(classified_edges_df, cycle_segments_df)
breakpoint_plot_df = workflow.build_breakpoint_plot_df(classified_edges_df)
cycle_plot_df = workflow.build_cycle_plot_df(cycles_df)

segment_plot_df.to_csv("segment_plot_data.csv", index=False)
breakpoint_plot_df.to_csv("breakpoint_plot_data.csv", index=False)
cycle_plot_df.to_csv("cycle_plot_data.csv", index=False)

pd.DataFrame({
    "table": ["segment_plot_data", "breakpoint_plot_data", "cycle_plot_data"],
    "rows": [len(segment_plot_df), len(breakpoint_plot_df), len(cycle_plot_df)]
})

,table,rows
0,segment_plot_data,13
1,breakpoint_plot_data,15
2,cycle_plot_data,2


In [7]:
valid_arcs_df = workflow.get_valid_arcs(breakpoint_plot_df)
ambiguous_breakpoints_df = breakpoint_plot_df[(breakpoint_plot_df["breakpoint_type"] == "unclear") | (breakpoint_plot_df["pos1"] < 0) | (breakpoint_plot_df["pos2"] < 0)]

display(segment_plot_df[["chromosome", "start", "end", "copy_number", "segment_id", "source"]].head(12))
display(valid_arcs_df[["raw_edge", "copy_number", "arc_start", "arc_end"]])
print(f"Ambiguous source breakpoint rows excluded from arc drawing: {len(ambiguous_breakpoints_df)}")

,chromosome,start,end,copy_number,segment_id,source
0,chr7,54729879,54743010,3.031595,<NA>,SequenceEdge
1,chr7,54743011,54830971,3.313981,<NA>,SequenceEdge
2,chr7,54830972,55194959,116.20461,1,SequenceEdge
3,chr7,55194960,55220100,3.167294,<NA>,SequenceEdge
4,chr7,55220101,55222712,3.180103,<NA>,SequenceEdge
5,chr7,55222713,55676816,116.217419,2,SequenceEdge
6,chr7,55676817,55677805,16.643945,<NA>,SequenceEdge
7,chr7,55677806,56117062,116.217419,4,SequenceEdge
8,chr7,56117063,56219661,3.326789,<NA>,SequenceEdge
9,chr7,54830972,55194959,116.20461,1,cycle_segments


,raw_edge,copy_number,arc_start,arc_end
14,chr7:56117062+->chr7:54830972-,112.890630,54830972,56117062
12,chr7:55222713-->chr7:55194959+,113.037316,55194959,55222713
13,chr7:55677806-->chr7:55676816+,99.573474,55676816,55677806


Ambiguous source breakpoint rows excluded from arc drawing: 4


## Final Visualization Code

The final figure uses blue horizontal lines for sequence-edge copy number, brown arcs for valid discordant breakpoints, and colored tracks below the x-axis for Cycle 1 and Cycle 2 segment membership.

In [8]:
workflow.save_final_figures(segment_plot_df, breakpoint_plot_df, cycle_plot_df)

output_figures = [
    Path("recreated_amplicon_plot.png"),
    Path("recreated_amplicon_plot.pdf"),
    Path("recreated_amplicon_plot_labeled.png"),
]

pd.DataFrame({"file": [path.name for path in output_figures], "exists": [path.exists() for path in output_figures]})

,file,exists
0,recreated_amplicon_plot.png,True
1,recreated_amplicon_plot.pdf,True
2,recreated_amplicon_plot_labeled.png,True


## Final Figure Preview

In [9]:
image = plt.imread("recreated_amplicon_plot.png")
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(image)
ax.axis("off")
plt.show()

/var/folders/l3/zf5fjtlx5712zkyflbpbyhr00000gn/T/ipykernel_72055/273381998.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Saved Outputs

In [10]:
output_files = [
    "graph_edges.csv",
    "cycle_segments.csv",
    "cycles.csv",
    "segment_plot_data.csv",
    "breakpoint_plot_data.csv",
    "cycle_plot_data.csv",
    "recreated_amplicon_plot.png",
    "recreated_amplicon_plot.pdf",
    "recreated_amplicon_plot_labeled.png",
]

pd.DataFrame({"file": output_files, "exists": [Path(file).exists() for file in output_files]})

,file,exists
0,graph_edges.csv,True
1,cycle_segments.csv,True
2,cycles.csv,True
3,segment_plot_data.csv,True
4,breakpoint_plot_data.csv,True
5,cycle_plot_data.csv,True
6,recreated_amplicon_plot.png,True
7,recreated_amplicon_plot.pdf,True
8,recreated_amplicon_plot_labeled.png,True


## Short Interpretation

The final plot reconstructs GBM39 Amplicon 1 as an ecDNA-focused amplicon structure on chr7. The horizontal blue sequence lines show genomic intervals placed at their estimated copy-number values. The high copy-number segments near 116 copies are the main amplified sequence blocks.

The brown arcs show the three valid intrachromosomal discordant breakpoint edges. These arcs connect distant positions on chr7 and represent structural rearrangements that help explain how the amplified segments are joined.

The cycle annotations summarize the two cycles found in the cycles file. Cycle 1 has copy count 99.57 and segment order `2+,4+,1+`. Cycle 2 has copy count 13.32 and segment order `3+,1+`. The colored tracks below the axis mark which genomic segments belong to each cycle.

The main assumptions are that source edges with `pos1 = -1` should not be drawn as arcs, and that concordant one-base adjacency edges represent sequence continuity rather than structural breakpoint arcs.